# Полный пайплайн распознавания эмоций на IEMOCAP (Audio+Video cascade)

**Схема:**
1. Вход: реплика IEMOCAP — отдельный `.wav` из `sentences/wav/...` и соответствующий диалоговый `.avi` с временными метками `[start, end]`.
2. **Audio branch** — Log-Mel + CNN: бинарная классификация *neutral vs emotional*.
3. Если `neutral` → итог = `neutral`.
4. Иначе → **Video branch** — кадры из `.avi` на интервале `[start, end]` → MediaPipe face crop (224×224) → EfficientNet-B0 → Attention Pooling → 8 классов RAVDESS → маппинг в 7 классов IEMOCAP (`calm → neutral`).

**Датасет:** все реплики IEMOCAP после фильтрации по 7 классам (`datasets/iemocap_processed/metadata.csv`).

**Метрики:** accuracy, F1 (macro), confusion matrix, доля ухода в видео-ветку, время на стадию, размер моделей.

In [1]:
import os, json, time, warnings, gc
warnings.filterwarnings('ignore')

import cv2
import librosa
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torchvision.models as tv_models
import torchvision.transforms as T

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))
print(f'Device: {device}')

IEMOCAP_ROOT = '../datasets/IEMOCAP_full_release'
IEMOCAP_META_CSV = '../datasets/iemocap_processed/metadata.csv'
AUDIO_CKPT_DIR = '../trained_models/audio_neutral_2'
VIDEO_CKPT_DIR = '../trained_models/video_emotion'
FACE_DETECTOR_PATH = '../raw_models/blaze_face_short_range.tflite'

IEMOCAP_LABELS = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']
IEMOCAP_L2I = {n: i for i, n in enumerate(IEMOCAP_LABELS)}

RAVDESS_LABELS = ['neutral', 'calm', 'happy', 'sad', 'angry', 'fearful', 'disgust', 'surprised']

RAVDESS_TO_IEMOCAP = {
    0: 0,  # neutral   → neutral
    1: 0,  # calm      → neutral
    2: 1,  # happy     → happy
    3: 2,  # sad       → sad
    4: 3,  # angry     → angry
    5: 4,  # fearful   → fearful
    6: 5,  # disgust   → disgust
    7: 6,  # surprised → surprised
}

def pytorch_size_mb(model):
    b = sum(p.numel() * p.element_size() for p in model.parameters())
    b += sum(x.numel() * x.element_size() for x in model.buffers())
    return b / 1024 ** 2

Device: mps


## 1. Загрузка метаданных IEMOCAP

Используем готовый `metadata.csv` из `iemocap.ipynb` — в нём уже применён маппинг эмоций в 7 классов, совместимых с RAVDESS.

In [2]:
meta = pd.read_csv(IEMOCAP_META_CSV)

def avi_path(row):
    return os.path.join(IEMOCAP_ROOT, row['session'], 'dialog', 'avi', 'DivX', f"{row['dialog']}.avi")

meta['avi_path'] = meta.apply(avi_path, axis=1)
meta['has_video'] = meta['avi_path'].apply(os.path.exists)
meta['has_audio'] = meta['audio_path'].apply(os.path.exists)
meta = meta[meta['has_audio']].reset_index(drop=True)

print(f'Всего реплик: {len(meta)} | Сессий: {meta["session"].nunique()} | С видео: {meta["has_video"].sum()}')
for name in IEMOCAP_LABELS:
    print(f'  {name:10s} {(meta["emotion"] == name).sum()}')

Всего реплик: 5680 | Сессий: 5 | С видео: 5680
  neutral    1708
  happy      1636
  sad        1084
  angry      1103
  fearful    40
  disgust    2
  surprised  107


## 2. Загрузка обученных моделей

Те же модели, что и в RAVDESS-пайплайне:
* **Audio:** Log-Mel (64×128) → `MelCNN` (бинарная, 0=neutral, 1=emotional)
* **Video:** 16 кадров → MediaPipe face crop (224×224) → EfficientNet-B0 → Attention Pooling → 8 классов RAVDESS

In [3]:
with open(os.path.join(AUDIO_CKPT_DIR, 'meta.json')) as f:
    audio_meta = json.load(f)
with open(os.path.join(VIDEO_CKPT_DIR, 'meta.json')) as f:
    video_meta = json.load(f)

SR = audio_meta['SR']; N_MELS = audio_meta['N_MELS']
MAX_LEN = audio_meta['MAX_LEN']; HOP = audio_meta['HOP']
MEL_MEAN = audio_meta['mel_mean']; MEL_STD = audio_meta['mel_std']

N_CLASSES = video_meta['N_CLASSES']
FEAT_DIM_EFF = video_meta['FEAT_DIM_EFF']
N_FRAMES = video_meta['N_FRAMES']
IMG_SIZE = video_meta['IMG_SIZE']
print(f'Audio meta: {audio_meta}')
print(f'Video meta: {video_meta}')

Audio meta: {'SR': 16000, 'N_MELS': 64, 'MAX_LEN': 128, 'HOP': 512, 'N_MFCC': 40, 'mel_mean': -33.17436218261719, 'mel_std': 27.39727783203125, 'hub_dim': 768}
Video meta: {'N_CLASSES': 8, 'FEAT_DIM': 576, 'FEAT_DIM_EFF': 1280, 'N_FRAMES': 16, 'IMG_SIZE': 224}


In [4]:
class MelCNN(nn.Module):
    def __init__(self, n_mels=64, max_len=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (n_mels // 8) * (max_len // 8), 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 2),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class AttentionPoolingClassifier(nn.Module):
    def __init__(self, feat_dim=1280, n_classes=8, hidden=128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, 64), nn.Tanh(), nn.Linear(64, 1),
        )
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(hidden, n_classes),
        )
    def forward(self, x):
        w = torch.softmax(self.attention(x), dim=1)
        pooled = (w * x).sum(dim=1)
        return self.classifier(pooled)

In [5]:
mel_cnn = MelCNN(N_MELS, MAX_LEN).to(device)
mel_cnn.load_state_dict(torch.load(os.path.join(AUDIO_CKPT_DIR, 'mel_cnn.pt'), map_location=device))
mel_cnn.eval()

efficientnet = tv_models.efficientnet_b0(weights=None)
efficientnet.classifier = nn.Identity()
eff_state = torch.load(os.path.join(VIDEO_CKPT_DIR, 'efficientnet_b0_backbone.pt'), map_location=device)
efficientnet.load_state_dict(eff_state)
efficientnet = efficientnet.to(device).eval()

attn_head = AttentionPoolingClassifier(FEAT_DIM_EFF, N_CLASSES).to(device)
attn_head.load_state_dict(torch.load(os.path.join(VIDEO_CKPT_DIR, 'head_efficientnet_attention.pt'), map_location=device))
attn_head.eval()

mel_cnn_mb = pytorch_size_mb(mel_cnn)
eff_mb = pytorch_size_mb(efficientnet)
head_mb = pytorch_size_mb(attn_head)
print(f'MelCNN: {mel_cnn_mb:.2f} MB')
print(f'EfficientNet-B0: {eff_mb:.2f} MB')
print(f'Attention head: {head_mb:.2f} MB')
print(f'Всего: {mel_cnn_mb + eff_mb + head_mb:.2f} MB')

MelCNN: 8.09 MB
EfficientNet-B0: 15.45 MB
Attention head: 0.94 MB
Всего: 24.48 MB


In [6]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_face_detector = mp_vision.FaceDetector.create_from_options(
    mp_vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path=FACE_DETECTOR_PATH),
        min_detection_confidence=0.5,
    )
)

imagenet_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
print('Face detector ready')

Face detector ready


I0000 00:00:1776713701.490405 22344896 gl_context.cc:407] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M1 Pro
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1776713701.495587 22344898 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


## 3. Инференс-утилиты

В IEMOCAP один `.avi` содержит двоих актёров в одном кадре — отрезаем половину экрана под говорящего (левая/правая) перед детекцией лица.

In [7]:
def load_audio_wav(wav_path, sr=SR):
    y, _ = librosa.load(wav_path, sr=sr, mono=True)
    return y


def audio_is_neutral(y):
    mel = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, hop_length=HOP)
    lm = librosa.power_to_db(mel, ref=np.max)
    if lm.shape[1] < MAX_LEN:
        lm = np.pad(lm, ((0, 0), (0, MAX_LEN - lm.shape[1])))
    else:
        lm = lm[:, :MAX_LEN]
    lm = (lm - MEL_MEAN) / (MEL_STD + 1e-8)
    xt = torch.tensor(lm[None, None], dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = mel_cnn(xt)
        prob = torch.softmax(logits, dim=1)[0].cpu().numpy()
    pred = int(logits.argmax(1).item())
    return pred == 0, float(prob[1])

In [8]:
def extract_frames_segment(avi_path, start_sec, end_sec, n_frames=N_FRAMES):
    cap = cv2.VideoCapture(avi_path)
    if not cap.isOpened():
        return []
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_f = max(0, int(start_sec * fps))
    end_f = min(total - 1, int(end_sec * fps))
    if end_f <= start_f:
        end_f = min(total - 1, start_f + 1)
    indices = np.linspace(start_f, end_f, n_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def speaker_side(utterance_id):
    """'L' или 'R' для говорящего.
    utterance_id: 'Ses01F_impro01_F000' — буква после номера сессии ('F')
    указывает пол «первого» актёра (слева); последняя буква перед номером
    реплики ('F' или 'M') — пол говорящего.
    """
    head = utterance_id.split('_')[0]          # 'Ses01F'
    first = head[-1]                           # 'F'
    speaker = utterance_id.split('_')[-1][0]   # 'F'
    return 'L' if speaker == first else 'R'


def crop_speaker_half(frame, speaker_lr):
    h, w = frame.shape[:2]
    mid = w // 2
    if speaker_lr == 'L':
        return frame[:, :mid]
    if speaker_lr == 'R':
        return frame[:, mid:]
    return frame


def detect_and_crop_face(frame, target_size=IMG_SIZE):
    mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = _face_detector.detect(mp_img)
    if result.detections:
        bb = result.detections[0].bounding_box
        h, w = frame.shape[:2]
        pad_x = int(bb.width * 0.1); pad_y = int(bb.height * 0.1)
        x1 = max(0, bb.origin_x - pad_x)
        y1 = max(0, bb.origin_y - pad_y)
        x2 = min(w, bb.origin_x + bb.width + pad_x)
        y2 = min(h, bb.origin_y + bb.height + pad_y)
        if x2 > x1 and y2 > y1:
            return cv2.resize(frame[y1:y2, x1:x2], (target_size, target_size))
    h, w = frame.shape[:2]; s = min(h, w)
    y1, x1 = (h - s) // 2, (w - s) // 2
    return cv2.resize(frame[y1:y1 + s, x1:x1 + s], (target_size, target_size))


def video_predict_emotion(avi_path, start_sec, end_sec, speaker_lr):
    frames = extract_frames_segment(avi_path, start_sec, end_sec, N_FRAMES)
    if not frames:
        crops = [np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)] * N_FRAMES
    else:
        while len(frames) < N_FRAMES:
            frames.append(frames[-1])
        crops = [detect_and_crop_face(crop_speaker_half(f, speaker_lr)) for f in frames[:N_FRAMES]]
    faces = torch.stack([imagenet_transform(c) for c in crops]).to(device)
    with torch.no_grad():
        feats = efficientnet(faces).unsqueeze(0)
        logits = attn_head(feats)
    return int(logits.argmax(1).item())

In [9]:
def full_pipeline(row):
    """Каскад: audio-neutral? → если нет, видео-классификатор.
    Предсказание возвращается в пространстве IEMOCAP (0..6)."""
    timings = {}
    t0 = time.perf_counter()
    y = load_audio_wav(row['audio_path'])
    timings['audio_load'] = time.perf_counter() - t0

    t0 = time.perf_counter()
    is_neutral, prob_emo = audio_is_neutral(y)
    timings['audio_infer'] = time.perf_counter() - t0

    if is_neutral or not row['has_video']:
        return {'pred': 0, 'routed_to_video': False,
                'prob_emotional': prob_emo, 'timings': timings}

    t0 = time.perf_counter()
    lr = speaker_side(row['utterance_id'])
    ravdess_pred = video_predict_emotion(row['avi_path'], row['start'], row['end'], lr)
    iemocap_pred = RAVDESS_TO_IEMOCAP[ravdess_pred]
    timings['video_infer'] = time.perf_counter() - t0
    return {'pred': iemocap_pred, 'routed_to_video': True,
            'prob_emotional': prob_emo, 'timings': timings}

## 4. Запуск по всему IEMOCAP

In [ ]:
N = len(meta)
preds = np.zeros(N, dtype=int)
routed = np.zeros(N, dtype=bool)
timings_audio_load = np.zeros(N)
timings_audio_infer = np.zeros(N)
timings_video_infer = np.full(N, np.nan)

y_true = meta['label'].values.astype(int)

t_total = time.perf_counter()
for i in tqdm(range(N), desc='Pipeline'):
    row = meta.iloc[i]
    r = full_pipeline(row)
    preds[i] = r['pred']
    routed[i] = r['routed_to_video']
    timings_audio_load[i] = r['timings']['audio_load']
    timings_audio_infer[i] = r['timings']['audio_infer']
    if r['routed_to_video']:
        timings_video_infer[i] = r['timings']['video_infer']
total_time = time.perf_counter() - t_total
print(f'Готово: {N} реплик за {total_time:.1f} сек ({total_time/N*1000:.1f} мс/реплика)')

Pipeline:   0%|          | 0/5680 [00:00<?, ?it/s]

## 5. Метрики

IEMOCAP — 7 классов (без `calm`). Видео-предсказания переведены из 8-классового RAVDESS-пространства (`calm → neutral`).

In [ ]:
present = sorted(set(y_true.tolist()) | set(preds.tolist()))
target_names = [IEMOCAP_LABELS[i] for i in present]

acc = accuracy_score(y_true, preds)
f1m = f1_score(y_true, preds, average='macro')
prec = precision_score(y_true, preds, average='macro', zero_division=0)
rec = recall_score(y_true, preds, average='macro', zero_division=0)
print(f'Accuracy : {acc:.4f}')
print(f'F1 macro : {f1m:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print()
print(classification_report(y_true, preds, labels=present, target_names=target_names, zero_division=0))

In [ ]:
n_video = int(routed.sum())
n_audio_only = len(routed) - n_video
print(f'Ушло в аудио-ветку (neutral):   {n_audio_only} ({n_audio_only/len(routed)*100:.1f}%)')
print(f'Ушло в видео-ветку (emotional): {n_video} ({n_video/len(routed)*100:.1f}%)')

y_true_bin = (y_true != 0).astype(int)
y_pred_bin = routed.astype(int)
gate_acc = accuracy_score(y_true_bin, y_pred_bin)
gate_f1 = f1_score(y_true_bin, y_pred_bin, average='macro')
print(f'\nКачество аудио-гейта (neutral vs emotional):')
print(f'  Accuracy: {gate_acc:.4f}  |  F1 macro: {gate_f1:.4f}')

mask_vid_correct_route = routed & (y_true_bin == 1)
if mask_vid_correct_route.sum() > 0:
    vid_acc = accuracy_score(y_true[mask_vid_correct_route], preds[mask_vid_correct_route])
    vid_f1 = f1_score(y_true[mask_vid_correct_route], preds[mask_vid_correct_route], average='macro')
    print(f'\nКачество видео-классификатора на корректно направленных репликах (N={int(mask_vid_correct_route.sum())}):')
    print(f'  Accuracy: {vid_acc:.4f}  |  F1 macro: {vid_f1:.4f}')

## 6. Визуализация

In [ ]:
cm = confusion_matrix(y_true, preds, labels=present)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[0], cbar=False)
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names, ax=axes[1], cbar=True, vmin=0, vmax=1)
axes[1].set_title('Confusion Matrix (row-normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
plt.tight_layout(); plt.show()

In [ ]:
audio_load_ms = timings_audio_load * 1000
audio_infer_ms = timings_audio_infer * 1000
video_infer_ms_only = timings_video_infer[~np.isnan(timings_video_infer)] * 1000

stages = ['Audio load\n(decode .wav)', 'Audio infer\n(MelCNN)', 'Video infer\n(EffNet+Attn)']
means = [audio_load_ms.mean(), audio_infer_ms.mean(),
         float(video_infer_ms_only.mean()) if len(video_infer_ms_only) else 0.0]
stds = [audio_load_ms.std(), audio_infer_ms.std(),
        float(video_infer_ms_only.std()) if len(video_infer_ms_only) else 0.0]

avg_pipeline_ms = audio_load_ms.mean() + audio_infer_ms.mean() + \
    (video_infer_ms_only.sum() / len(meta) if len(video_infer_ms_only) else 0.0)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = ['#3498db', '#2ecc71', '#e67e22']
bars = axes[0].bar(stages, means, yerr=stds, capsize=6, color=colors, alpha=0.85)
axes[0].set_ylabel('мс / реплика')
axes[0].set_title(f'Время на стадию (средн.)  |  Avg pipeline: {avg_pipeline_ms:.1f} мс')
axes[0].grid(axis='y', alpha=0.3)
for b, v in zip(bars, means):
    axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() + max(means) * 0.02,
                 f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

per_total_ms = audio_load_ms + audio_infer_ms + np.nan_to_num(timings_video_infer * 1000, nan=0.0)
axes[1].hist(per_total_ms, bins=40, color='#9b59b6', alpha=0.85, edgecolor='white')
axes[1].axvline(per_total_ms.mean(), color='red', linestyle='--',
                label=f'mean {per_total_ms.mean():.1f} ms')
axes[1].set_xlabel('мс / реплика'); axes[1].set_ylabel('count')
axes[1].set_title('Распределение времени полного пайплайна')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie([n_audio_only, n_video],
            labels=[f'Neutral (audio only)\n{n_audio_only}',
                    f'Emotional → Video\n{n_video}'],
            colors=['#95a5a6', '#e67e22'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Маршрутизация запросов')

model_names = ['MelCNN\n(audio)', 'EfficientNet-B0\n(backbone)', 'Attention head\n(video)']
sizes = [mel_cnn_mb, eff_mb, head_mb]
bars = axes[1].bar(model_names, sizes, color=['#2ecc71', '#3498db', '#e74c3c'], alpha=0.85)
axes[1].set_ylabel('MB')
axes[1].set_title(f'Размер моделей  |  Всего: {sum(sizes):.1f} MB')
axes[1].grid(axis='y', alpha=0.3)
for b, v in zip(bars, sizes):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + max(sizes) * 0.02,
                 f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
summary = pd.DataFrame([{
    'Реплик всего': len(meta),
    'Accuracy (7-cls)': f'{acc:.4f}',
    'F1 macro (7-cls)': f'{f1m:.4f}',
    'Audio-gate Acc': f'{gate_acc:.4f}',
    'Audio-gate F1': f'{gate_f1:.4f}',
    '% → Video': f'{n_video/len(routed)*100:.1f}%',
    'Avg pipeline (ms)': f'{avg_pipeline_ms:.1f}',
    'Total size (MB)': f'{sum(sizes):.1f}',
}])
print(summary.to_string(index=False))